# 01 — Data Pull

Probe APIs then fetch historical + current season data for the 2024-25 Gallagher Premiership playoffs.

**Source:** TheSportsDB free tier (league ID 4414 — English Premiership Rugby)  
**Coverage:** 71 completed matches across 5 seasons (2020-21 → 2024-25)  
**Note:** ESPN returned 400 for all league IDs; TSDB free tier is capped at ~15 unique events per season.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.data_pull import probe_espn, probe_thesportsdb

try:
    espn = probe_espn()
    print('ESPN OK:', list(espn.keys())[:5])
except Exception as e:
    print('ESPN FAILED:', e)

try:
    tsdb = probe_thesportsdb()
    leagues = [l.get('strLeague', '') for l in (tsdb.get('countrys') or [])]
    print('TSDB Rugby leagues:', [l for l in leagues if 'rugby' in l.lower() or 'premier' in l.lower()])
except Exception as e:
    print('TSDB FAILED:', e)

## Fetch historical data

TSDB league ID 4414 (English Premiership Rugby) returns ~15 unique completed results per season.
ESPN returns 400 for the Premiership league (league ID 270559 not served by their rugby-union endpoint).

In [ ]:
from src.data_pull import fetch_historical_tsdb, compute_team_stats, save_raw, SEMIFINALISTS
import pandas as pd

historical = fetch_historical_tsdb()  # uses TSDB_LEAGUE_ID = 4414
print(f'Total: {len(historical)} matches across seasons: {historical["season"].unique()}')
print(historical.head(10))

In [ ]:
# Current season stats (most recent season)
current_season = historical[historical['season'] == historical['season'].max()]
print(f'Current season matches: {len(current_season)}')

team_stats = [compute_team_stats(current_season, t) for t in SEMIFINALISTS]
for s in team_stats:
    print(s)

save_raw(historical, team_stats)
print('Saved to data/raw/')

In [ ]:
# Verify saved files
h = pd.read_csv('../data/raw/historical_matches.csv')
s = pd.read_csv('../data/raw/current_season_stats.csv')
print('historical_matches.csv:', h.shape, '| columns:', list(h.columns))
print('
current_season_stats.csv:')
print(s.to_string(index=False))